In [ ]:
!pip --quiet install ijson
!pip --quiet install -U weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.8 MB/s eta 0:00:00


In [ ]:
import ijson # For reading json file
import pandas as pd
from pathlib import Path
import numpy as np
import json
import weaviate
from weaviate.classes.init import Auth
from weaviate.util import generate_uuid5
import os
import weaviate.classes.config as wc

from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments, PreTrainedModel, PreTrainedTokenizerFast


# Set up keys here :3
# weaviate_url = user_secrets.get_secret("WEAVIATE_URL")
# weaviate_api_key = user_secrets.get_secret("WEAVIATE_API_KEY")
# huggingface_key = user_secrets.get_secret("HUGGINGFACE_APIKEY")
weaviate_url = ###Replace with your API###
weaviate_api_key = ###Replace with your API###
huggingface_key = ###Replace with your API###
# print(huggingface_key)
headers = {
    "X-HuggingFace-Api-Key": huggingface_key,
}
# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=Auth.api_key(weaviate_api_key),
    headers=headers,
)
anony=None
print(client.is_ready())
# except:
#     anony = "must"
#     # print("Cannot connect to database")

True


In [ ]:
!gdown 1sApSMYsk5ld3fgIl9o-pJqWGX6WpkQ9G

Downloading...
From (original): https://drive.google.com/uc?id=1sApSMYsk5ld3fgIl9o-pJqWGX6WpkQ9G
From (redirected): https://drive.google.com/uc?id=1sApSMYsk5ld3fgIl9o-pJqWGX6WpkQ9G&confirm=t&uuid=f3fb508b-0f96-4846-bf98-f4c087712eb6
To: /content/allMeSH_2022.zip
100% 8.92G/8.92G [03:13<00:00, 46.0MB/s]


In [ ]:
!unzip "/content/allMeSH_2022.zip" -d "/content/"

Archive:  /content/allMeSH_2022.zip
  inflating: /content/allMeSH_2022.json  


# Semantic Chunking

In [ ]:
!pip --quiet install "chonkie[all]"
!pip --quiet install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/1

In [ ]:
from chonkie import SemanticChunker,SentenceChunker

# Basic initialization with default parameters
chunker = SemanticChunker(
    embedding_model="minishlab/potion-base-8M",  # Default model
    threshold="auto",                               # Similarity threshold (0-1) or (1-100) or "auto"
    # similarity_window = 2,
    chunk_size=512,                              # Maximum tokens per chunk
    min_sentences=1                              # Initial sentences per chunk
)

# # Basic initialization with default parameters
# chunker = SentenceChunker(
#     tokenizer_or_token_counter="gpt2",                # Supports string identifiers
#     chunk_size=512,                                   # Maximum tokens per chunk
#     chunk_overlap=0,                                  # Overlap between chunks
#     min_sentences_per_chunk=1,                        # Minimum sentences in each chunk
#     min_characters_per_sentence=12,                   # Minimum characters per sentence
#     approximate=True,                                 # Use approximate token counting
#     delim=[".", "?", "!", "\n"],                      # Delimiters to use for chunking
#     include_delim="prev",                             # Include the delimiter in the chunk
#     return_type="chunks"                              # Return Chunks or texts only
# )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/30.2M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/271k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/684k [00:00<?, ?B/s]

In [ ]:
text = "Worldwide, hypertensive disorders of pregnancy (HDP), fetal growth restriction (FGR) and preterm birth remain the leading causes of maternal and fetal pregnancy-related mortality and (long-term) morbidity. Fetal cardiac deformation changes can be the first sign of placental dysfunction, which is associated with HDP, FGR and preterm birth. In addition, preterm birth is likely associated with changes in electrical activity across the uterine muscle. Therefore, fetal cardiac function and uterine activity can be used for the early detection of these complications in pregnancy. Fetal cardiac function and uterine activity can be assessed by two-dimensional speckle-tracking echocardiography (2D-STE), non-invasive fetal electrocardiography (NI-fECG), and electrohysterography (EHG). This study aims to generate reference values for 2D-STE, NI-fECG and EHG parameters during the second trimester of pregnancy and to investigate the diagnostic potential of these parameters in the early detection of HDP, FGR and preterm birth.METHODS: In this longitudinal prospective cohort study, eligible women will be recruited from a tertiary care hospital and a primary midwifery practice. In total, 594 initially healthy pregnant women with an uncomplicated singleton pregnancy will be included. Recordings of NI-fECG and EHG will be made weekly from 22 until 28 weeks of gestation and 2D-STE measurements will be performed 4-weekly at 16, 20, 24 and 28 weeks gestational age. Retrospectively, pregnancies complicated with pregnancy-related diseases will be excluded from the cohort. Reference values for 2D-STE, NI-fECG and EHG parameters will be assessed in uncomplicated pregnancies. After, 2D-STE, NI-fCG and EHG parameters measured during gestation in complicated pregnancies will be compared with these reference values.DISCUSSION: This will be the a large prospective study investigating new technologies that could potentially have a high impact on antepartum fetal monitoring."

In [ ]:
chunks = chunker.chunk(text)

for chunk in chunks:
    print(f"Chunk text: {chunk.text}")
    print(f"Token count: {chunk.token_count}")
    print(f"Number of sentences: {len(chunk.sentences)}")
    print(f"{chunk.start_index} - {chunk.end_index}")

Chunk text: Worldwide, hypertensive disorders of pregnancy (HDP), fetal growth restriction (FGR) and preterm birth remain the leading causes of maternal and fetal pregnancy-related mortality and (long-term) morbidity. Fetal cardiac deformation changes can be the first sign of placental dysfunction, which is associated with HDP, FGR and preterm birth. In addition, preterm birth is likely associated with changes in electrical activity across the uterine muscle. Therefore, fetal cardiac function and uterine activity can be used for the early detection of these complications in pregnancy.
Token count: 119
Number of sentences: 4
0 - 579
Chunk text:  Fetal cardiac function and uterine activity can be assessed by two-dimensional speckle-tracking echocardiography (2D-STE), non-invasive fetal electrocardiography (NI-fECG), and electrohysterography (EHG). This study aims to generate reference values for 2D-STE, NI-fECG and EHG parameters during the second trimester of pregnancy and to investigat

In [ ]:
# Check if the collection exists before attempting to create it
if not client.collections.exists(name="MedicalArticles"):
    client.collections.create(
        name="MedicalArticles",
        properties=[
            wc.Property(name="pmid", data_type=wc.DataType.TEXT),
            wc.Property(name="title", data_type=wc.DataType.TEXT),
            wc.Property(name="abstractText", data_type=wc.DataType.TEXT),
            wc.Property(name="journal", data_type=wc.DataType.TEXT),
            wc.Property(name="year", data_type=wc.DataType.TEXT),
            wc.Property(name="meshMajor", data_type=wc.DataType.TEXT_ARRAY),
            wc.Property(name="chunk", data_type=wc.DataType.TEXT),
            wc.Property(name="chunkid", data_type=wc.DataType.INT)
        ],
        # Define the vectorizer module (none, as we will add our own vectors)
        vectorizer_config=wc.Configure.Vectorizer.none(),
        # Define the generative module
        generative_config=wc.Configure.Generative.cohere()
    )
else:
    print("Collection 'MedicalArticles' already exists.")

In [ ]:
def process_text(text):
    # Xóa "BACKGROUND: " từ đầu văn bản
    if text.startswith("BACKGROUND: "):
        text = text[12:]  # 12 là độ dài của "BACKGROUND: "

    # Tìm và cắt văn bản trước "TRIAL REGISTRATION"
    trial_index = text.find("TRIAL REGISTRATION")
    if trial_index != -1:
        text = text[:trial_index].strip()

    return text

In [ ]:
# 3. Hàm xử lý và import dữ liệu
def process_and_import_articles(file_path, client, batch_size=10000):
    collection = client.collections.get("MedicalArticles")
    if not collection:
        raise ValueError("Collection 'MedicalArticles' not found.")
    with open(file_path, encoding="latin-1") as file:
        parser = ijson.parse(file)
        current_article = {}
        batch_articles = []

        with collection.batch.dynamic() as batch:
            for prefix, event, value in parser:
                if prefix == 'articles.item' and event == 'start_map':
                    current_article = {}

                elif prefix == 'articles.item' and event == 'end_map':
                    if current_article and 'abstractText' in current_article:
                        try:
                            # Xử lý văn bản
                            current_article['abstractText'] = process_text(current_article['abstractText'])

                            # Chunking
                            chunks = chunker.chunk(current_article['abstractText'])
                            i = 1
                            for chunk in chunks:
                                # Tạo embedding cho abstractText
                                vector = model.encode(chunk.text)

                                # Chuẩn bị object để import
                                properties = {
                                    "pmid": current_article.get('pmid', ''),
                                    "title": current_article.get('title', ''),
                                    "abstractText": current_article.get('abstractText', ''),
                                    "journal": current_article.get('journal', ''),
                                    "year": current_article.get('year', ''),
                                    "meshMajor": current_article.get('meshMajor', []),
                                    "chunk": chunk.text,
                                    "chunkid": i
                                }

                                # Add vào batch
                                batch.add_object(
                                    properties=properties,
                                    uuid=generate_uuid5(current_article['pmid']),
                                    vector=vector.tolist()
                                )
                                i = i + 1
                        except Exception as e:
                            print(f"Error processing article {current_article.get('pmid')}: {str(e)}")

                    current_article = {}

                elif event == 'map_key':
                    key = value
                    if key == 'meshMajor':
                        current_article[key] = []

                elif prefix.startswith('articles.item.'):
                    if event == 'string' and not prefix.endswith('.item'):
                        current_article[key] = value
                    elif prefix.endswith('.item'):
                        current_article[key].append(value)

In [ ]:
try:
    # Process và import dữ liệu

    # Chunking and passing to Sentence Transformer
    from sentence_transformers import SentenceTransformer, util
    import torch

    # 1. Load a pretrained Sentence Transformer model
    model = SentenceTransformer("all-MiniLM-L6-v2", device = "cuda")
    process_and_import_articles(
        file_path='/content/allMeSH_2022.json',
        client=client
    )

    print("Data import completed successfully!")

except Exception as e:
    print(f"Error: {str(e)}")
finally:
    client.close()